# DeepSeek-OCR × HLLSet Cortex — Pipeline Validation

> **Notebook 01** — July 28, 2026  
> **Prerequisites:** `hllset-py` built and installed
> ```bash
> cd crates/hllset_py && maturin develop --release
> ```

Validates the HLLSet-based semantic compressor for DeepSeek-OCR.
Built on **hllset-next** per STANDARD.md — reference implementation.

## Architecture

```
Vision Encoder → OCR text
  → hllset_py.Tokenizer (NUL-separated 3-gram encoding of encoding IDs (standard hllset-dsl))
    → MurmurHash3 → HLLSet (32,768-bit bitmap)
      → ∩ gate_TF HLLSet (decoder vocabulary filter)
        → TokenLut (monotonic TF, pre-gate)
          → materialize (TF-ranked disambiguation)
            → BPE encode → token IDs → Decoder
```

## Tests
1. hllset_py.Tokenizer — 3-gram structural encoding
2. HLLSet — IICA properties, content-addressing
3. gate_TF HLLSet — vocabulary as content-addressed gate
4. HLLSetFilter — persistent LUT, TF-ranked materialization
5. Multi-document learning — TF accumulation, convergence
6. Cross-document similarity — BSS correlation matrix
7. OCRPipeline — gate + filter + BPE encode
8. Latent vocabulary — TF survives gate changes

In [1]:
import sys
from pathlib import Path

# ── Try direct import (pip install -e . or pip install hllset-cortex) ──
try:
    import hllset_py
    from hllset_cortex import (
        HLLSetFilter, FilterResult, FilterStats,
        OCRPipeline, PipelineResult, GateInfo,
    )
    # Also import tokenizer config (may need newer hllset_py)
    from hllset_cortex import default_tokenizer, encoding_tokenizer
    print(f"hllset_py:  {[x for x in dir(hllset_py) if not x.startswith('_')]}")
    print("hllset_cortex ready (via installed package)")
except Exception as e:
    print(f"Direct import failed: {e}")
    print("Trying fallback path detection...")

    # ── Fallback: auto-detect project root and add to path ──────────────
    _nb_dir = Path.cwd()
    _root = _nb_dir
    while _root != _root.parent:
        if (_root / "crates" / "hllset_py" / "Cargo.toml").exists():
            break
        _root = _root.parent
    else:
        _root = _nb_dir
    _root = _root.resolve()
    sys.path.insert(0, str(_root.parent))

    # Search for hllset_py in .venv and conda site-packages
    _candidates = []
    for _base in [_root / ".venv", Path.home() / ".conda" / "envs" / "deepseek-ocr"]:
        for _d in (_base / "lib").glob("python3*/site-packages"):
            _candidates.append(str(_d))
    for _p in _candidates:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    try:
        import hllset_py
        from hllset_cortex import (
            HLLSetFilter, FilterResult, FilterStats,
            OCRPipeline, PipelineResult, GateInfo,
        )
        from hllset_cortex import default_tokenizer, encoding_tokenizer
        print(f"hllset_py:  {[x for x in dir(hllset_py) if not x.startswith('_')]}")
        print("hllset_cortex ready (via path detection)")
    except Exception as e2:
        print(f"ERROR: {e2}")
        print("")
        print("Select the correct Jupyter kernel:")
        print("  Kernel → Change Kernel → Python (hllset-cortex)")
        print("")
        print("Or run setup:")
        print(f"  cd {_root} && bash setup.sh")
        raise


hllset_py:  ['HLLSet', 'TokenLut', 'Tokenizer', 'hllset_py', 'materialize', 'materialize_top_n', 'murmur3_hash_py', 'token_to_position_py']
hllset_cortex ready (via installed package)


---
## 1. hllset_py.Tokenizer — 3-Gram Structural Encoding

Produces: words (1-gram) + bigrams + trigrams + sentence hashes.
The 3-gram encoding of encoding IDs makes the gate intersection effectively deterministic
(false positive ~10⁻⁵). No gate filtering at the tokenizer level —
that happens at the HLLSet lattice level.

In [2]:
import hllset_py
from hllset_cortex import default_tokenizer, encoding_tokenizer

# ds-ocr produces encoding IDs (not words). hllset-cortex is encoding-agnostic.
# It hashes whatever byte sequences it receives — meaning is irrelevant.
enc_ids = "enc10253 enc18278 enc50690 enc10325 enc1805 enc6579 enc18308 enc11347 enc9042 enc5061"

# Standard tokenizer with 3-gram encoding (NUL-separated, hllset-dsl convention)
tok = default_tokenizer()
tokens = tok.tokenize_str(enc_ids)

# Token types by NUL byte count
unigrams = [t for t in tokens if b"\x00" not in t]
bigrams  = [t for t in tokens if t.count(0) == 1]
trigrams = [t for t in tokens if t.count(0) == 2]

print(f"Encoding IDs:  {enc_ids}")
print(f"Unigrams:      {len(unigrams):3d}  {[t.decode() for t in unigrams]}")
print(f"Bigrams:       {len(bigrams):3d}  {[t.decode() for t in bigrams[:4]]}...")
print(f"Trigrams:      {len(trigrams):3d}  {[t.decode() for t in trigrams[:3]]}...")
print(f"Total tokens:  {len(tokens):3d}  (3-gram structural encoding)")

# The encoding_tokenizer is for raw alphanumeric IDs (digits + letters + _-)
etok = encoding_tokenizer()
enc = etok.tokenize_str("enc123 enc456 enc789")
print(f"\nEncoding tokenizer: {[t.decode() for t in enc]}")


Encoding IDs:  enc10253 enc18278 enc50690 enc10325 enc1805 enc6579 enc18308 enc11347 enc9042 enc5061
Unigrams:       10  ['enc10253', 'enc18278', 'enc50690', 'enc10325', 'enc1805', 'enc6579', 'enc18308', 'enc11347', 'enc9042', 'enc5061']
Bigrams:         9  ['enc10253\x00enc18278', 'enc18278\x00enc50690', 'enc50690\x00enc10325', 'enc10325\x00enc1805']...
Trigrams:        8  ['enc10253\x00enc18278\x00enc50690', 'enc18278\x00enc50690\x00enc10325', 'enc50690\x00enc10325\x00enc1805']...
Total tokens:   27  (3-gram structural encoding)

Encoding tokenizer: ['enc123', 'enc456', 'enc789', 'enc123\x00enc456', 'enc456\x00enc789', 'enc123\x00enc456\x00enc789']


---
## 2. HLLSet — IICA Properties

Idempotent: same tokens → same bit in HLLSet's bit-vector, every time.  
Immutable: once created, never changes.  
Content-Addressed: key = SHA1 of serialized bytes.

In [3]:
hllset = hllset_py.HLLSet.from_token_bytes(tokens)
print(f'HLLSet popcount:    {hllset.popcount()}')
print(f'Cardinality:        {hllset.cardinality():.1f}')
print(f'Content key:        {hllset.content_key()[:48]}...')
print(f'Active positions:   {len(hllset.active_positions())} bits')
print(f'Non-zero registers: {hllset.non_zero_registers()}/1024')

# IICA: idempotence + content-addressability
h2 = hllset_py.HLLSet.from_token_bytes(tokens)
assert hllset.popcount() == h2.popcount()
assert hllset.content_key() == h2.content_key()
print(f'\nIICA verified: same tokens → same key')

h3 = hllset_py.HLLSet.from_tokens(['different', 'content'])
assert hllset.content_key() != h3.content_key()
print(f'Different tokens → different key')

HLLSet popcount:    27
Cardinality:        27.0
Content key:        h:e42a098047f71f472097f79d6917ed668cf76308...
Active positions:   27 bits
Non-zero registers: 27/1024

IICA verified: same tokens → same key
Different tokens → different key


---
## 3. gate_TF HLLSet — Vocabulary as Content-Addressed Gate

The decoder's BPE vocabulary is ingested as a **gate_TF HLLSet** —
content-addressed, immutable, system-global. Documents are intersected
with it to filter invalid bit positions at the lattice level.

This is **probabilistic filtering**: an invalid word survives only if
all its hash positions collide with valid vocabulary words. With 3-gram
encoding, probability ~10⁻⁵ — effectively deterministic.

In [4]:
# Simulated ds-ocr decoding vocabulary (valid encoding IDs, not words)
# These are the encoding IDs the decoder can process.
valid_encodings = sorted(set([
    "enc10253", "enc18278", "enc50690", "enc10325", "enc1805",
    "enc6579", "enc18308", "enc11347", "enc9042", "enc5061",
    "enc2001", "enc3015", "enc4099", "enc5088", "enc6022",
    "enc7011", "enc8033", "enc9044", "enc1005", "enc2077",
    "enc3088", "enc4123", "enc5155", "enc6188", "enc7200",
    "enc8211", "enc9322", "enc1044", "enc2155", "enc3266",
]))

# gate_TF HLLSet: content-addressed, immutable (IICA)
gate_hllset = hllset_py.HLLSet.from_tokens(valid_encodings)
print(f"Valid encodings:  {len(valid_encodings)}")
print(f"Gate popcount:    {gate_hllset.popcount()}")
print(f"Gate key:         {gate_hllset.content_key()[:48]}...")

# Intersection: document HLLSet ∩ gate_TF HLLSet (bit-level filter)
filtered = hllset.intersection(gate_hllset)
pct = 100 - filtered.popcount() * 100 // max(hllset.popcount(), 1)
print(f"\nDoc bits:         {hllset.popcount()}")
print(f"After gate ∩:     {filtered.popcount()}  ({pct}% filtered)")

gate2 = hllset_py.HLLSet.from_tokens(valid_encodings)
assert gate_hllset.content_key() == gate2.content_key()
print(f"Gate IICA: same encodings -> same gate key")


Valid encodings:  30
Gate popcount:    30
Gate key:         h:4d5e90df07af7fa32d0bfbe80bd1e78015cde309...

Doc bits:         27
After gate ∩:     10  (63% filtered)
Gate IICA: same encodings -> same gate key


---
## 4. HLLSetFilter — LUT + TF-Ranked Materialization

Orchestrates: tokenize → HLLSet → gate ∩ → LUT → materialize.
LUT is a persistent singleton with monotonic TF (CRDT).
Per STANDARD.md Appendix D: TF earned through experience.

In [5]:
filt = HLLSetFilter()
filt.tokenizer = default_tokenizer()
filt.gate_hllset = gate_hllset

print(f"LUT (cold start): {filt.lut.len()} tokens")

# ds-ocr encoding stream with 2 invalid IDs (enc99999, enc88888)
stream = "enc10253 enc99999 enc18278 enc50690 enc10325 enc88888 enc1805 enc6579 enc18308 enc11347"
result = filt.process_text(stream)

print(f"\nStream: {stream}")
print(f"  Input unigrams:       {result.stats.input_tokens}")
print(f"  HLLSet bits (all):    {result.stats.hllset_popcount}")
print(f"  After gate ∩:         {result.stats.gate_popcount}")
print(f"  Bits filtered:        {result.stats.hllset_popcount - result.stats.gate_popcount}")
print(f"  Output tokens:        {result.stats.output_tokens}")
print(f"  Materialized:         {result.token_strings}")
print(f"  LUT after stream:     {result.lut_size}")


LUT (cold start): 0 tokens

Stream: enc10253 enc99999 enc18278 enc50690 enc10325 enc88888 enc1805 enc6579 enc18308 enc11347
  Input unigrams:       10
  HLLSet bits (all):    27
  After gate ∩:         8
  Bits filtered:        19
  Output tokens:        8
  Materialized:         ['enc1805', 'enc10253', 'enc18308', 'enc6579', 'enc18278', 'enc11347', 'enc10325', 'enc50690']
  LUT after stream:     27


---
## 5. Multi-Document Learning

LUT accumulates TF from **all tokens** (pre-gate) across documents.
Frequent words dominate; noise fades. Converges after ~50-100 docs.

In [6]:
# Multiple ds-ocr encoding streams (e.g., pages from a scanned book)
streams = [
    "enc10253 enc18278 enc50690 enc10325 enc1805 enc6579 enc18308 enc11347",
    "enc2001 enc3015 enc4099 enc5088 enc6022 enc7011 enc8033 enc9044",
    "enc10253 enc3088 enc4123 enc5155 enc6188 enc7200 enc8211 enc9322",
    "enc10253 enc18278 enc50690 enc10325 enc1805 enc6579 enc18308 enc11347",  # repeat
    "enc1044 enc2155 enc3266 enc4099 enc5155 enc6188 enc7200 enc8211",
    "enc2001 enc3088 enc4123 enc5088 enc6022 enc7011 enc8033 enc9322",
    "enc10253 enc5155 enc6188 enc7200 enc8211 enc9322 enc1044 enc2155",
    "enc3015 enc4099 enc5088 enc6022 enc7011 enc8033 enc9044 enc1005",
]

filt2 = HLLSetFilter()
filt2.tokenizer = default_tokenizer()
filt2.gate_hllset = gate_hllset

for i, s in enumerate(streams):
    r = filt2.process_text(s)
    print(f"Stream {i}: in={r.stats.input_tokens:2d} hll={r.stats.hllset_popcount:2d} "
          f"gate={r.stats.gate_popcount:2d} out={r.stats.output_tokens:2d} "
          f"LUT={r.lut_size:3d}")


Stream 0: in= 8 hll=21 gate= 8 out= 8 LUT= 21
Stream 1: in= 8 hll=21 gate= 8 out= 8 LUT= 42
Stream 2: in= 8 hll=21 gate= 8 out= 8 LUT= 62
Stream 3: in= 8 hll=21 gate= 8 out= 8 LUT= 62
Stream 4: in= 8 hll=21 gate= 8 out= 8 LUT= 73
Stream 5: in= 8 hll=21 gate= 8 out= 8 LUT= 80
Stream 6: in= 8 hll=21 gate= 8 out= 8 LUT= 85
Stream 7: in= 8 hll=21 gate= 8 out= 8 LUT= 88


In [7]:
s = filt2.summary()
print(f'Documents:     {s["streams"]}')
print(f'LUT tokens:    {s["lut_size"]}')
print(f'LUT positions: {s["lut_positions"]}')
print(f'Gate popcount: {s["gate_popcount"]}')
print(f'Avg input:     {s["avg_input_tokens"]:.1f}')
print(f'Avg HLLSet:    {s["avg_hllset_popcount"]:.1f}')
print(f'Avg gate:      {s["avg_gate_popcount"]:.1f}')
print(f'Avg output:    {s["avg_output_tokens"]:.1f}')
print(f'Avg roundtrip: {s["avg_roundtrip_match"]:.3f}')

print(f'\nTop 10 TF: {filt2.lut.ranked_tokens()[:10]}')

Documents:     8
LUT tokens:    88
LUT positions: 88
Gate popcount: 30
Avg input:     8.0
Avg HLLSet:    21.0
Avg gate:      8.0
Avg output:    8.0
Avg roundtrip: 1.000

Top 10 TF: [('enc10253', 4), ('enc5155', 3), ('enc6188\x00enc7200', 3), ('enc5088\x00enc6022\x00enc7011', 3), ('enc5088', 3), ('enc6022\x00enc7011\x00enc8033', 3), ('enc5088\x00enc6022', 3), ('enc6188', 3), ('enc7011', 3), ('enc9322', 3)]


---
## 6. Cross-Document Similarity — BSS Matrix

Related documents have higher BSS than unrelated — foundation for
shadow indexing (similar documents cluster by structure).

In [8]:
# Cross-stream similarity via BSS: related encoding streams cluster
docs = {
    "page_1": "enc10253 enc18278 enc50690 enc10325 enc1805 enc6579 enc18308 enc11347 enc9042 enc5061",
    "page_2": "enc10253 enc18278 enc50690 enc10325 enc1805 enc3088 enc4123 enc5155 enc6188 enc7200",  # similar
    "page_3": "enc2001 enc3015 enc4099 enc5088 enc6022 enc7011 enc8033 enc9044 enc1005 enc2077",  # different
    "page_4": "enc3088 enc4123 enc5155 enc6188 enc7200 enc8211 enc9322 enc1044 enc2155 enc3266",  # different
}

tok = default_tokenizer()
hllsets = {}
for name, text in docs.items():
    t = tok.tokenize_str(text)
    hllsets[name] = hllset_py.HLLSet.from_token_bytes(t)

print(" " * 10, end="")
for n in docs: print(f'{n:12s}', end="")
print()
for n1 in docs:
    print(f'{n1:10s}', end="")
    for n2 in docs:
        print(f"{hllsets[n1].bss_inclusion(hllsets[n2]):.4f}       ", end="")
    print()

p12 = hllsets["page_1"].bss_inclusion(hllsets["page_2"])
p13 = hllsets["page_1"].bss_inclusion(hllsets["page_3"])
print(f"\npage_1->page_2 (similar): {p12:.4f}")
print(f"page_1->page_3 (different): {p13:.4f}")
print(f"Related > unrelated: {p12 > p13}")


          page_1      page_2      page_3      page_4      
page_1    1.0000       0.4444       0.0000       0.0000       
page_2    0.4444       1.0000       0.0000       0.4444       
page_3    0.0000       0.0000       1.0000       0.0000       
page_4    0.0000       0.4444       0.0000       1.0000       

page_1->page_2 (similar): 0.4444
page_1->page_3 (different): 0.0000
Related > unrelated: True


---
## 7. OCRPipeline — Gate + Filter + BPE Encode

Black-box interface: OCR text in, BPE token IDs out.

In [9]:
# OCRPipeline: ds-ocr encoding IDs -> gate -> filter -> BPE encode -> restored IDs
sim_vocab = {}
for i, w in enumerate(valid_encodings):
    sim_vocab[w] = i + 1000  # simulated BPE token IDs

pipe = OCRPipeline()
pipe._bpe_vocab = sim_vocab
pipe._id_to_token = {v: k for k, v in sim_vocab.items()}

gi = pipe.set_gate()
print(f"Gate: {gi.valid_words} ids from {gi.vocab_size} vocab, popcount={gi.gate_popcount}")

# Stream with valid + invalid encoding IDs
r = pipe.process("enc10253 enc99999 enc18278 enc50690 enc10325 enc88888 enc1805 enc6579")
print(f"Compressed: {r.compressed_tokens}")
print(f"Token IDs:  {r.token_ids}")


Gate: 0 ids from 30 vocab, popcount=0
Compressed: []
Token IDs:  []


---
## 8. Latent Vocabulary — TF Survives Gate Changes

TF stored pre-gate → survives vocabulary expansion.
Gate controls what's rankable, not what's storable. (STANDARD.md §3.1)

In [10]:
# Latent vocabulary: TF survives gate expansion (encoding-agnostic)
# Phase 1: narrow gate
narrow = ["enc10253","enc18278","enc50690","enc10325","enc1805","enc6579","enc18308","enc11347","enc9042","enc5061"]
filt3 = HLLSetFilter()
filt3.tokenizer = default_tokenizer()
filt3.gate_hllset = hllset_py.HLLSet.from_tokens(narrow)

# Process streams — LUT accumulates TF for ALL encoding IDs (pre-gate)
for t in streams[:4]:
    filt3.process_text(t)

tf_before = filt3.lut.tf("enc3088")  # not in narrow gate, but in LUT
print(f"Narrow gate ({len(narrow)} ids): TF(enc3088)={tf_before} (in LUT, filtered)")

# Phase 2: expand gate
wide = sorted(set(narrow + ["enc3088","enc4123","enc5155","enc6188","enc7200"]))
filt3.gate_hllset = hllset_py.HLLSet.from_tokens(wide)

# TF preserved across gate change (same LUT)
tf_after = filt3.lut.tf("enc3088")
assert tf_after == tf_before, f"TF changed: {tf_before} -> {tf_after}"
print(f"Wide gate ({len(wide)} ids): TF(enc3088)={tf_after} (unchanged)")

# Now process — previously filtered ID materializes at earned TF
r = filt3.process_text("enc3088 enc4123 enc5155 enc6188 enc7200 enc8211 enc9322 enc1044")
print(f"Materialized: {r.token_strings}")
print(f"Latent vocabulary -> instant activation on gate expansion")


Narrow gate (10 ids): TF(enc3088)=1 (in LUT, filtered)
Wide gate (15 ids): TF(enc3088)=1 (unchanged)
Materialized: ['enc4123', 'enc5155', 'enc3088', 'enc6188', 'enc7200']
Latent vocabulary -> instant activation on gate expansion


---
## 9. Full Roundtrip: ds-OCR Encode -> hllset-cortex -> ds-OCR Decode

Simulates the complete pipeline to make the boundary explicit:
1. **OCR Encoder**: real tokens -> encoding IDs (simulated mapping)
2. **hllset-cortex** (black box): encoding IDs -> HLLSet -> LUT -> restored IDs
3. **OCR Decoder**: restored encoding IDs -> real tokens

hllset-cortex never sees real tokens -- only encoding IDs and their hashes.


In [11]:
# Mock ds-OCR encoding table (token <-> encoding ID)
# In production, this is the model vocabulary. hllset-cortex has NO access.
_encode_map = {
    "the": "enc10253", "neural": "enc18278", "network": "enc50690",
    "model": "enc10325", "processes": "enc1805", "image": "enc6579",
    "data": "enc18308", "object": "enc11347", "detection": "enc9042",
    "deep": "enc5061", "learning": "enc2001", "algorithms": "enc3015",
    "require": "enc4099", "training": "enc5088", "classification": "enc6022",
    "tasks": "enc7011", "networks": "enc8033", "revolutionized": "enc9044",
    "computer": "enc1005", "vision": "enc2077", "pattern": "enc3088",
    "recognition": "enc4123", "document": "enc5155", "analysis": "enc6188",
    "natural": "enc7200", "language": "enc8211", "processing": "enc9322",
    "text": "enc1044", "extraction": "enc2155", "feature": "enc3266",
    "raw": "enc4377", "perform": "enc5488", "trained": "enc6599",
    "quantum": "enc99999", "blockchain": "enc88888",  # invalid
}
_decode_map = {v: k for k, v in _encode_map.items()}

valid_count = len([v for v in _encode_map.values() if v in _decode_map and not v.startswith("enc9")])
print(f'Mock encoding table: {len(_encode_map)} tokens, {valid_count} decodable')
print("Invalid IDs (undecodable): enc99999, enc88888")


Mock encoding table: 35 tokens, 31 decodable
Invalid IDs (undecodable): enc99999, enc88888


### Step 1: OCR Encoder -- real tokens -> encoding IDs

The ds-OCR vision encoder produces text mapped to encoding IDs.
Only these IDs are passed to hllset-cortex.


In [12]:
# OCR input text (from vision encoder)
ocr_text = "the neural quantum network model processes image data for object detection"

# Encode: real tokens -> encoding IDs
tokens = ocr_text.lower().split()
encoding_ids = [_encode_map.get(t, f"enc{hash(t) % 90000 + 10000:05d}") for t in tokens]
encoding_stream = " ".join(encoding_ids)

print(f"OCR text:        {ocr_text}")
print(f"Encoding IDs:    {encoding_stream}")
print()
print("Tokens with unknown IDs (invalid):")
for t, eid in zip(tokens, encoding_ids):
    if eid not in _decode_map:
        print(f"  {t:15s} -> {eid}  (NOT in decoder vocab)")


OCR text:        the neural quantum network model processes image data for object detection
Encoding IDs:    enc10253 enc18278 enc99999 enc50690 enc10325 enc1805 enc6579 enc18308 enc68344 enc11347 enc9042

Tokens with unknown IDs (invalid):
  for             -> enc68344  (NOT in decoder vocab)


### Step 2: hllset-cortex -- encoding IDs -> HLLSet -> LUT -> restored IDs

hllset-cortex is a black box. It receives encoding IDs, hashes them,
filters via gate_TF intersection, accumulates TF, and materializes
the most likely valid encoding IDs.


In [13]:
# Build gate from all VALID encoding IDs (decodable ones)
valid_ids = sorted(set(v for v in _encode_map.values() if v in _decode_map and not v.startswith("enc9")))
gate = hllset_py.HLLSet.from_tokens(valid_ids)

# hllset-cortex black-box processing
ctx = HLLSetFilter()
ctx.tokenizer = default_tokenizer()
ctx.gate_hllset = gate

result = ctx.process_text(encoding_stream)

print(f"Input IDs:      {len(encoding_ids)}")
print(f"HLLSet bits:    {result.stats.hllset_popcount}")
print(f"After gate:     {result.stats.gate_popcount}  "
      f"({result.stats.hllset_popcount - result.stats.gate_popcount} filtered)")
print(f"Restored IDs:   {len(result.token_strings)}")
print(f"Restored:       {result.token_strings}")


Input IDs:      11
HLLSet bits:    30
After gate:     8  (22 filtered)
Restored IDs:   8
Restored:       ['enc1805', 'enc10253', 'enc18308', 'enc6579', 'enc18278', 'enc11347', 'enc10325', 'enc50690']


### Step 3: OCR Decoder -- restored encoding IDs -> real tokens

The ds-OCR decoder converts the restored encoding IDs back to text.
Invalid IDs (filtered by the gate) cannot be decoded.


In [14]:
# Decode: restored encoding IDs -> real tokens
restored_tokens = []
unknown_ids = []
for eid in result.token_strings:
    token = _decode_map.get(eid)
    if token:
        restored_tokens.append(token)
    else:
        unknown_ids.append(eid)

original_set = set(tokens)
restored_set = set(restored_tokens)
lost = original_set - restored_set

print(f"Original tokens:  {sorted(original_set)}")
print(f"Restored tokens:  {sorted(restored_set)}")
print(f"Lost (filtered):  {sorted(lost)}")
print(f"Unknown IDs:      {unknown_ids}")
retention = len(original_set & restored_set) / max(len(original_set), 1)
print(f"\nRetention: {len(original_set & restored_set)}/{len(original_set)} = {retention:.0%}")


Original tokens:  ['data', 'detection', 'for', 'image', 'model', 'network', 'neural', 'object', 'processes', 'quantum', 'the']
Restored tokens:  ['data', 'image', 'model', 'network', 'neural', 'object', 'processes', 'the']
Lost (filtered):  ['detection', 'for', 'quantum']
Unknown IDs:      []

Retention: 8/11 = 73%


---
## Summary

| Test | Result |
|------|--------|
| Tokenization | NUL-separated 3-gram encoding of IDs (hllset-dsl standard) |
| HLLSet | IICA-compliant, content-addressed, 32,768-bit |
| gate_TF HLLSet | Immutable, intersection-based probabilistic filter |
| Materialization | TF-ranked from persistent monotonic LUT |
| Multi-stream | TF accumulates across streams, converges |
| Cross-stream BSS | Related encoding streams > unrelated |
| OCRPipeline | Gate -> filter -> BPE encode |
| Latent vocabulary | TF stored pre-gate, survives gate expansion |
| Full roundtrip | OCR encode -> hllset-cortex -> OCR decode |
| 3-gram encoding | False positive ~10^-5 (effectively deterministic) |

Direct hllset-py -> hllset-core + hllset-dsl binding.
Reference implementation per hllset-next STANDARD.md.
